# ☕ Aula 03 — Smart Contracts e Tokenização RWA

**Missão:** acompanhar um lote de café, criar regras automáticas e dividir sua representação digital.  
**Ambiente:** Google Colab, sem instalação e sem carteira de criptomoedas.  
**Duração:** 60 min de conceitos + 40 min de prática.

> O notebook simula uma blockchain para tornar visíveis estado, permissões, eventos, hashes e tokens. Ele não movimenta dinheiro real.


## 1. O caso: Café Aurora

Uma cooperativa possui um lote premium de **1.000 kg**, avaliado didaticamente em **R$ 100.000**. Produtor, transportadora, inspetor e comprador querem consultar o mesmo histórico.

Perguntas da aula:

- Quem pode registrar ou alterar cada etapa?
- Como detectar uma tentativa de adulteração?
- O que um token representa no mundo real?
- Quando blockchain oferece vantagem sobre um banco de dados?


## 2. Smart contract em linguagem simples

Pense em um **cartório digital programável**:

1. alguém envia um pedido;
2. o contrato verifica identidade e regra;
3. uma operação válida muda o estado;
4. o contrato emite um evento;
5. uma operação inválida preserva o estado anterior.

### Um pequeno trecho de Solidity

```solidity
string public status = "Na fazenda";

function embarcar() public {
    status = "Em transporte";
    emit StatusAlterado(status);
}

event StatusAlterado(string novoStatus);
```

Na prática abaixo, Python representará as mesmas três ideias: **estado**, **função** e **evento**.


In [ ]:
# Célula 1 — personagens e estado inicial
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from hashlib import sha256
import json

PAPEIS = {
    'Alice': 'produtor',
    'Bruno': 'transportadora',
    'Carla': 'inspetor',
    'Diego': 'comprador',
    'Mallory': 'visitante'
}

print('🎭 Participantes')
for pessoa, papel in PAPEIS.items():
    print(f'  {pessoa:8} → {papel}')


In [ ]:
# Célula 2 — nosso smart contract simulado
class ContratoCafe:
    def __init__(self):
        self.lotes = {}
        self.eventos = []

    def _evento(self, tipo, dados):
        anterior = self.eventos[-1]['hash'] if self.eventos else 'GENESIS'
        evento = {
            'n': len(self.eventos) + 1,
            'tipo': tipo,
            'dados': dados,
            'anterior': anterior,
            'instante': datetime.now(timezone.utc).isoformat()
        }
        texto = json.dumps(evento, sort_keys=True, ensure_ascii=False)
        evento['hash'] = sha256(texto.encode()).hexdigest()[:16]
        self.eventos.append(evento)
        print(f"📣 EVENTO {evento['n']}: {tipo} | hash {evento['hash']}")

    def registrar(self, ator, lote_id, descricao, custodiante):
        if PAPEIS.get(ator) != 'produtor':
            raise PermissionError('apenas o produtor registra lotes')
        if lote_id in self.lotes:
            raise ValueError('lote já existe')
        self.lotes[lote_id] = {
            'descricao': descricao, 'status': 'Na fazenda',
            'custodiante': custodiante, 'qualidade': None
        }
        self._evento('LoteRegistrado', {'lote': lote_id, 'por': ator})

    def transferir(self, ator, lote_id, novo_custodiante, novo_status):
        lote = self.lotes[lote_id]
        if ator != lote['custodiante']:
            raise PermissionError('apenas o custodiante atual pode transferir')
        lote['custodiante'] = novo_custodiante
        lote['status'] = novo_status
        self._evento('CustodiaTransferida', {'lote': lote_id, 'de': ator, 'para': novo_custodiante, 'status': novo_status})

    def inspecionar(self, ator, lote_id, nota):
        if PAPEIS.get(ator) != 'inspetor':
            raise PermissionError('apenas o inspetor publica a nota')
        if not 0 <= nota <= 100:
            raise ValueError('nota deve ficar entre 0 e 100')
        self.lotes[lote_id]['qualidade'] = nota
        self._evento('QualidadePublicada', {'lote': lote_id, 'nota': nota, 'por': ator})

contrato = ContratoCafe()
print('🤖 Contrato Café Aurora criado')


## 3. Rodada 1 — o produtor registra o lote

Alice cria o ativo digital que representa o lote físico. O identificador `CAFE-001` liga o contrato ao cadastro do sistema corporativo.


In [ ]:
contrato.registrar(
    ator='Alice',
    lote_id='CAFE-001',
    descricao='1.000 kg de café arábica premium',
    custodiante='Alice'
)
contrato.lotes['CAFE-001']


## 4. Rodada 2 — a viagem e a inspeção

A cadeia de custódia avança. Cada comando gera um evento, semelhante a um recibo que outros sistemas podem consumir.


In [ ]:
contrato.transferir('Alice', 'CAFE-001', 'Bruno', 'Em transporte')
contrato.inspecionar('Carla', 'CAFE-001', 92)
contrato.transferir('Bruno', 'CAFE-001', 'Diego', 'Entregue')

print('\n📦 Estado atual')
for campo, valor in contrato.lotes['CAFE-001'].items():
    print(f'{campo:12}: {valor}')


## 5. Desafio — o falso transportador

Mallory tenta alterar o lote. Execute a célula e observe duas coisas: a mensagem de rejeição e a preservação do estado.


In [ ]:
estado_antes = contrato.lotes['CAFE-001'].copy()
try:
    contrato.transferir('Mallory', 'CAFE-001', 'Mallory', 'Desviado')
except PermissionError as erro:
    print('❌ Transação rejeitada:', erro)

estado_depois = contrato.lotes['CAFE-001'].copy()
print('✅ Estado preservado?', estado_antes == estado_depois)


## 6. Auditoria — a corrente de eventos

Cada evento contém o hash anterior. Se alguém adulterar um registro, a corrente deixa de fechar. Essa é uma versão didática do encadeamento visto na Aula 01.


In [ ]:
def auditar(eventos):
    esperado = 'GENESIS'
    for evento in eventos:
        if evento['anterior'] != esperado:
            return False, evento['n']
        esperado = evento['hash']
    return True, None

for e in contrato.eventos:
    print(f"#{e['n']} {e['tipo']:22} anterior={e['anterior']:16} hash={e['hash']}")

valida, ponto = auditar(contrato.eventos)
print('\n🔍 Corrente válida?', valida)


## 7. Tokenização RWA

Vamos representar o valor didático de R$ 100.000 com 10.000 tokens fungíveis. Cada token representa **1/10.000 da posição econômica definida pelo projeto**.

> O código calcula frações. Custódia, resgate, vínculo jurídico, identidade e conformidade precisam existir fora dele.


In [ ]:
VALOR_LOTE = 100_000
TOTAL_TOKENS = 10_000
saldos = {'Cooperativa': 8_000, 'Ana': 500, 'Beto': 1_500}

def extrato(nome):
    tokens = saldos[nome]
    percentual = tokens / TOTAL_TOKENS * 100
    valor_proporcional = tokens / TOTAL_TOKENS * VALOR_LOTE
    barra = '🟪' * round(percentual / 5) + '⬜' * (20 - round(percentual / 5))
    return nome, tokens, percentual, valor_proporcional, barra

for nome in saldos:
    nome, tokens, pct, valor, barra = extrato(nome)
    print(f'{nome:12} {tokens:5} tokens | {pct:5.1f}% | R$ {valor:,.2f} | {barra}')

assert sum(saldos.values()) == TOTAL_TOKENS
print('\n✅ Oferta total conferida:', sum(saldos.values()))


## 8. Miniatividade em duplas

Escolha um ativo: equipamento, obra de arte, ingresso, energia gerada ou recebível. Preencha:

1. **Ativo:** o que existe no mundo real?
2. **Token:** o que cada unidade representa?
3. **Atores:** quem registra, transfere e verifica?
4. **Oráculo:** qual informação externa entra no contrato?
5. **Risco:** que mentira o contrato não consegue descobrir sozinho?
6. **Escolha técnica:** banco de dados, blockchain privada ou rede pública? Por quê?

### Entrega

- notebook executado;
- resposta às seis perguntas;
- uma modificação no simulador, como novo status, ator ou regra;
- explicação da tentativa inválida executada.


## 9. Síntese

- Um **smart contract** executa regras compartilhadas.
- Um **evento** comunica uma mudança a outros sistemas.
- Um **token** representa algo definido pelo arranjo técnico e jurídico.
- Um **RWA** depende de custódia e informação do mundo real.
- Blockchain faz mais sentido quando várias organizações precisam compartilhar confiança.

## Referências

- SOLIDITY. *Solidity Documentation*. <https://docs.soliditylang.org/en/latest/>. Acesso em 30 set. 2026.
- VOGELSTELLER, F.; BUTERIN, V. *ERC-20 Token Standard*. <https://eips.ethereum.org/EIPS/eip-20>.
- ENTRIKEN, W. et al. *ERC-721 Non-Fungible Token Standard*. <https://eips.ethereum.org/EIPS/eip-721>.
- RADOMSKI, W. et al. *ERC-1155 Multi Token Standard*. <https://eips.ethereum.org/EIPS/eip-1155>.
- OPENZEPPELIN. *Contracts Documentation*. <https://docs.openzeppelin.com/contracts/>.
- BUTERIN, V. *Ethereum Whitepaper*. <https://ethereum.org/en/whitepaper/>.
- ANTONOPOULOS, A.; WOOD, G. *Mastering Ethereum*. O'Reilly, 2018. <https://github.com/ethereumbook/ethereumbook>.

**Nota:** a simulação serve para aprendizagem. Ela não implementa consenso, assinatura digital real, rede distribuída nem garantias de produção.
